In [0]:
from pyspark.sql.functions import current_timestamp, col, max

source_db = "hive_metastore.we_retail_hub"
target_db = "hive_metastore.we_retail_mart"

tables = spark.catalog.listTables(source_db)

for table in tables:
    table_name = table.name
    source_table = f"{source_db}.{table_name}"
    target_table = f"{target_db}.{table_name}"

    print(f"Loading {source_table} -> {target_table}")

    try:
        df = spark.read.table(source_table)

        # Skip tables without audit column (optional safety)
        if "aud_load_timestamp" not in df.columns:
            print(f"Skipping {table_name}: no aud_load_timestamp column")
            continue

        # Get latest load timestamp
        max_ts = df.agg(max("aud_load_timestamp")).collect()[0][0]

        # Keep only latest batch
        df_latest = df.filter(col("aud_load_timestamp") == max_ts)

        # Recreate audit timestamp for mart
        df_final = df_latest.drop("aud_upd_timestamp")
        df_final = df_final.drop("active_flag")
        df_final = df_final.drop("aud_load_timestamp").withColumn("aud_load_timestamp", current_timestamp())

        # FULL REFRESH with schema evolution enabled
        df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

        print(f"Done: {table_name}")

    except Exception as e:
        print(f"Failed: {table_name} -> {e}")

print("All tables processed.")